## Imports

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

## Define the methods to:
* Fetch the tickers current prices, EPS and trainling PE
* Get the historic close prices to calculate change

In [2]:
def get_ticker_details(ticker_symbol):
    try:
        ticker_obj = yf.Ticker(ticker_symbol)
        info = ticker_obj.info
        
        # Extracting the requested fields
        price = info.get('currentPrice') or info.get('regularMarketPrice')
        currency = info.get('currency', 'Unknown')
        eps = info.get('trailingEps')  # Earnings Per Share
        pe_ratio = info.get('trailingPE')  # P/E Ratio
        sector = info.get('sector', 'N/A')
        
        return pd.Series([price, currency, eps, pe_ratio, sector])
    
    except Exception as e:
        # Returning None for all columns if the fetch fails
        return pd.Series([None, "Error", None, None, None])

In [3]:
def get_last_prices_dict(ticker_column):
    """
    Takes a pandas Series of tickers and returns a dictionary 
    mapping {Ticker: Last_Close_Price}.
    """
    # 1. Get unique tickers as a list
    unique_tickers = ticker_column.unique().tolist()
    
    # 2. Fetch last 1 day of data
    # We use 'Close' specifically to simplify the returned object
    data = yf.download(unique_tickers, period="1d", progress=False)['Close']
    
    # 3. Handle data structure based on ticker count
    # yfinance returns a Series for 1 ticker and a DataFrame for 2+
    if isinstance(data, pd.DataFrame):
        # Take the last available row and convert to dict
        return data.iloc[-1].to_dict()
    else:
        # It's a Series (single ticker), return dict with the first ticker name
        return {unique_tickers[0]: data.iloc[-1]}

In [4]:
def get_pct_change_dict(ticker_column):
    """
    Fetches the last 2 sessions for a Series of tickers and 
    returns a dictionary mapping {Ticker: Percentage_Change}.
    """
    unique_tickers = ticker_column.unique().tolist()
    
    # Fetch 2 days to get (Today's Close - Yesterday's Close) / Yesterday's Close
    # We use 'Close' only to keep the object lightweight
    data = yf.download(unique_tickers, period="2d", progress=False)['Close']
    
    # Calculate percentage change between the two rows (axis=0 is the default)
    # .pct_change() calculates: (Current - Previous) / Previous
    pct_changes = data.pct_change().iloc[-1] # Grab only the most recent calculation
    
    # Handle single ticker (Series) vs Multiple tickers (DataFrame)
    if isinstance(pct_changes, pd.Series):
        return pct_changes.to_dict()
    else:
        # If only 1 ticker was passed, pct_changes is a float
        return {unique_tickers[0]: pct_changes}

### Import ticker list

In [5]:
df_dividend = pd.read_csv('dividend.csv')

### Fetch current prince, EPS and PE information

In [6]:
print("Fetching prices... please wait.")
cols = ['Current_Price', 'Currency', 'EPS', 'PE_Ratio', 'Sector']
df_dividend[cols] = df_dividend['Ticker'].apply(get_ticker_details)

# 4. Display the results
print("Done!")
#print(df_dividend)

Fetching prices... please wait.


Done!


### Calculate price changes

In [7]:
# 1. Get the mapping dictionary
change_map = get_pct_change_dict(df_dividend['Ticker'])

# 2. Map it to a new column (multiplied by 100 for percentage format)
df_dividend['Pct_Change'] = df_dividend['Ticker'].map(change_map) * 100

/tmp/ipykernel_2416/3610519473.py:14: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pct_changes = data.pct_change().iloc[-1] # Grab only the most recent calculation


### Sort and format dataframe

In [8]:
df_sorted = df_dividend.sort_values(by = ["Sector", "PE_Ratio"])
df_sorted[['PE_Ratio', 'Pct_Change']] = df_sorted[['PE_Ratio', 'Pct_Change']].round(1)
df_sorted['Sector'] = df_sorted['Sector'].str[:3].str.upper()

rename_cols =  {
    'Current_Price': 'Price',
    'PE_Ratio': 'PE',
    'Pct_Change': '% Chg',
    'Currency': 'Cur.'
}

print(df_sorted)

df_sorted.rename(columns=rename_cols, inplace=True)

print(df_sorted)

      Ticker  Current_Price Currency     EPS  PE_Ratio Sector  Pct_Change
15  MORLD.OL          17.30      NOK    1.59      10.9    ENE        -0.7
7     ENH.OL           8.96      NOK    0.39      23.0    ENE         1.6
3    NONG.OL         156.16      NOK   15.29      10.2    FIN         0.3
2     DNB.OL         301.80      NOK   28.40      10.6    FIN         0.2
10   MING.OL         204.30      NOK   19.16      10.7    FIN        -1.1
19      ITUB           8.38      USD    0.77      10.9    FIN         0.0
11  SB1NO.OL         211.00      NOK   16.99      12.4    FIN         0.2
9     B2I.OL          23.90      NOK    1.68      14.2    FIN         2.4
4     STB.OL         176.20      NOK   11.70      15.1    FIN         1.1
8     GJF.OL         254.00      NOK   12.55      20.2    FIN         0.8
13   MPCC.OL          22.90      NOK    5.17       4.4    IND         0.1
12    WWI.OL         736.00      NOK  151.38       4.9    IND         1.5
6    WAWI.OL         125.20      NOK  

### Format HTML report and save

In [9]:
# Create the table HTML string
table_html = df_sorted.to_html(
    index=False, 
    classes='table table-striped table-hover table-sm', # Bootstrap classes
    border=0,
    justify='unset'
)

In [10]:
def color_change(val):
    color = 'green' if val > 0 else 'red'
    return f'color: {color}; font-weight: bold'

In [11]:
styled_html = (
    df_sorted.style
    .map(color_change, subset=['% Chg'])
    .format({
        'Price': '{:.2f}',      # Force 2 decimals
        'EPS': '{:.2f}',      # Force 2 decimals
        'PE': '{:.1f}',      # Force 1 decimal
        '% Chg': '{:+.1f}%'   # Force 1 decimal, a '+' sign for pos numbers, and a % symbol
    })
    .set_table_attributes('class="table table-striped table-hover"')
    .hide(axis='index')
    .to_html()
)

In [12]:
# Inject styled_html into your template
html_template = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sector Report</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {{ padding: 5px; background-color: #f8f9fa; }}
        .table-responsive {{ border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); background: white; margin: 0;}}
        h2 {{ color: #343a40; margin-bottom: 20px; font-weight: bold; }}
        /* Ensure the pandas-generated styles don't conflict with Bootstrap */
        table {{ width: 100%; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Sector Performance Report</h2>
        <div class="table-responsive">
            {styled_html}
        </div>
        <p class="text-muted mt-3">Last updated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}</p>
    </div>
</body>
</html>
"""

# Save the full report
with open("stock_report.html", "w") as f:
    f.write(html_template)